In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy  as np

import tensorflow as tf
import keras_nlp

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score,
)

from utils import (
    load_data, plot_distribution, plot_text_lengths,
    plot_accuracy, plot_loss, plot_confusion_matrix,
    plot_precision_recall, load_tokenizer,
)

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/project/backend'

DATA_DIR      = os.path.join(PROJECT_ROOT, 'Data', 'Generated')
WEIGHTS_DIR   = os.path.join(PROJECT_ROOT, 'Trained_Weights')

os.makedirs(WEIGHTS_DIR, exist_ok=True)


print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('WEIGHTS_DIR  :', WEIGHTS_DIR)

In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'Using GPU: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('Using CPU')

In [ ]:
CFG = dict(
    model_name   = 'deberta_v3_base_en',   
    embedding_name = 'microsoft/deberta-v3-base',
    max_length   = 512,
    batch_size   = 8,          
    epochs       = 5,
    lr           = 2e-5,
    warmup_ratio = 0.1,
    val_split    = 0.15,
    test_split   = 0.10,
    num_labels   = 2,
    weight_decay = 0.01,
)

In [ ]:
VALID_CSV = os.path.join(DATA_DIR, 'Chunk1_A','train.csv')

df = load_data(VALID_CSV)
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

In [ ]:
# Label distribution from utils
plot_distribution(df, 'label')

In [ ]:
plot_text_lengths(df)

In [ ]:
df['label'] = df['label'].astype(int)

df['text'] = (
    df['INSTRUCTION'].astype(str) + '\n' +
    df['reasoning'].astype(str)   + '\n' +
    df['RESPONSE'].astype(str)
)

print(f'Total samples after cleaning: {len(df)}')
print('Sample text (first 200 chars):')
print(df['text'].iloc[0][:200])

In [ ]:
train_df, temp_df = train_test_split(
    df[['text', 'label']],
    test_size   = CFG['val_split'] + CFG['test_split'],
    stratify    = df['label'],
    random_state= SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size   = CFG['test_split'] / (CFG['val_split'] + CFG['test_split']),
    stratify    = temp_df['label'],
    random_state= SEED,
)

print(f'Train : {len(train_df):>5} rows')
print(f'Val   : {len(val_df):>5} rows')
print(f'Test  : {len(test_df):>5} rows')

In [ ]:
tokenizer = load_tokenizer(CFG["embedding_name"])
print('Tokenizer loaded:', CFG["embedding_name"])
print('Vocab size:', tokenizer.vocab_size)

In [ ]:
def tokenize_texts(texts, labels, tokenizer, max_length, batch_size, shuffle=False):
    token_ids = tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='np', 
    )
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            'token_ids':    tf.constant(token_ids['input_ids'],      dtype=tf.int32),
            'padding_mask': tf.constant(token_ids['attention_mask'], dtype=tf.int32),
        },
        tf.constant(list(labels), dtype=tf.int32),
    ))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(texts), seed=SEED)
    return dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = tokenize_texts(
    train_df['text'], train_df['label'],
    tokenizer, CFG['max_length'], CFG['batch_size'], shuffle=True
)
val_ds = tokenize_texts(
    val_df['text'], val_df['label'],
    tokenizer, CFG['max_length'], CFG['batch_size']
)
test_ds = tokenize_texts(
    test_df['text'], test_df['label'],
    tokenizer, CFG['max_length'], CFG['batch_size']
)

print(f'Train batches : {len(train_ds)}')
print(f'Val   batches : {len(val_ds)}')
print(f'Test  batches : {len(test_ds)}')

In [ ]:
def load_model(model_name, num_labels=2):
    model = keras_nlp.models.DebertaV3Classifier.from_preset(model_name, preprocessor=None,num_classes=num_labels)
    return model

model = load_model(CFG['model_name'], num_labels=CFG['num_labels'])
model.summary()

In [ ]:
total_steps  = len(train_ds) * CFG['epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])


lr_schedule = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate = CFG['lr'],
    decay_steps           = total_steps - warmup_steps,
    end_learning_rate     = 0.0,
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate = lr_schedule,
    weight_decay  = CFG['weight_decay'],
)

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer= optimizer,
    loss= loss_fn,
    metrics= [tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)

print(f'Total training steps : {total_steps}')
print(f'Warmup steps         : {warmup_steps}')

In [ ]:
best_model_path = os.path.join(WEIGHTS_DIR, 'deberta_reasoning_best.keras')

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath          = best_model_path,
        monitor           = 'val_accuracy',
        save_best_only    = True,
        save_weights_only = False,
        verbose           = 1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor              = 'val_accuracy',
        patience             = 3,
        restore_best_weights = True,
        verbose              = 1,
    )
]

history_obj = model.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = CFG['epochs'],
    callbacks       = callbacks,
)

history = {
    'train_loss': history_obj.history['loss'],
    'val_loss':   history_obj.history['val_loss'],
    'train_acc':  history_obj.history['accuracy'],
    'val_acc':    history_obj.history['val_accuracy'],
}

print(f'Best Val Accuracy: {max(history["val_acc"]):.4f}')
print(f'Checkpoint: {best_model_path}')

In [ ]:
# Training curves
plot_accuracy(history['train_acc'], history['val_acc'])
plot_loss(history['train_loss'],    history['val_loss'])

In [ ]:
# Predict on test set
logits_list, labels_list = [], []
for x_batch, y_batch in test_ds:
    logits = model(x_batch, training=False)
    logits_list.append(logits.numpy())
    labels_list.append(y_batch.numpy())

all_logits  = np.concatenate(logits_list, axis=0)
test_labels = np.concatenate(labels_list, axis=0)
test_probs  = tf.nn.softmax(all_logits, axis=-1).numpy()[:, 1]
test_preds  = np.argmax(all_logits, axis=-1)

test_acc = accuracy_score(test_labels, test_preds)
print(f'Test Accuracy : {test_acc:.4f}')
print(f'Test ROC-AUC  : {roc_auc_score(test_labels, test_probs):.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, labels=[1], target_names=['Valid']))

In [ ]:
# Confusion matrix
plot_confusion_matrix(test_labels, test_preds, classes=['Invalid (0)', 'Valid (1)'])

In [ ]:
# Precision-Recall / F1 curve 
plot_precision_recall(test_labels, test_probs)

In [ ]:
# Save full final model in Keras format
model.save_weights(os.path.join(WEIGHTS_DIR, 'deberta_reasoning_final_weights.h5'))
print(f'Final model weights saved to: {WEIGHTS_DIR}')